### Model Training

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
%pip install catboost
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load the dataset
train = pd.read_csv("../data/processed/featured_train.csv")
test = pd.read_csv("../data/processed/featured_test.csv")

#### Prepare Features and Target

In [3]:
test_ids = test["ID"]

# Features
X = train.drop(columns=["ID", "label"])
y = train["label"]

# Remove ID from test data
X_test = test.drop(columns=["ID"])

#### Train-Validation Split

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_valid.shape[0]}")

Training samples: 1456
Validation samples: 365


##### Verify the class distribution

In [5]:
print("Training Set")
print(y_train.value_counts(normalize=True) * 100)

print()

print("Validation Set")
print(y_valid.value_counts(normalize=True) * 100)

Training Set
label
0    59.615385
1    40.384615
Name: proportion, dtype: float64

Validation Set
label
0    59.726027
1    40.273973
Name: proportion, dtype: float64


#### Train the Baseline CatBoost Model

In [6]:
# Initialize the CatBoost classifier
catboost_model = CatBoostClassifier(
    random_state=42,
    verbose=0
)

# Train the model
catboost_model.fit(X_train, y_train)

CatBoostClassifier(random_state=42, verbose=0)

#### Evaluate the Baseline CatBoost Model

In [7]:
# Binary predictions (default threshold = 0.5)
y_pred = catboost_model.predict(X_valid)

# Probability predictions
y_prob = catboost_model.predict_proba(X_valid)[:, 1]

#### Calculate Metrics

In [8]:
# Calculate evaluation metrics
f1 = f1_score(y_valid, y_pred)

roc_auc = roc_auc_score(y_valid, y_prob)

competition_score = (0.6 * f1) + (0.4 * roc_auc)

#### Display results

In [9]:
print("CatBoost Baseline Performance")
print("-" * 35)

print(f"F1-Score         : {f1:.4f}")
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"Competition Score: {competition_score:.4f}")

CatBoost Baseline Performance
-----------------------------------
F1-Score         : 0.9759
ROC-AUC          : 0.9967
Competition Score: 0.9842


#### Confusion Matrix

In [10]:
cm = confusion_matrix(y_valid, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

cm_df

,Predicted 0,Predicted 1
Actual 0,216,2
Actual 1,5,142


In [13]:
sample_submission = pd.read_csv("../data/raw/SampleSubmission.csv")
sample_submission.head()

,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,0,0
1,ID_TS_NEW_7SPRN3PB,0,0
2,ID_TS_NEW_LZOWPHLC,0,0
3,ID_TS_NEW_DN6TUF64,0,0
4,ID_TS_NEW_95N82M49,0,0


#### Retrain Catboost on Full Dataset

In [15]:
# Retrain model using all training data
catboost_model.fit(X, y)
print("Final model trained successfully!")

Final model trained successfully!


#### Save the model

In [11]:
catboost_model.save_model(
    "../models/catboost_baseline_model.cbm"
)

print("Model saved!")

Model saved!


#### Generate test predictions

In [12]:
# Binary predictions
test_pred = catboost_model.predict(X_test)

# Probability predictions
test_prob = catboost_model.predict_proba(X_test)[:,1]

#### Create Submission File

In [13]:
submission = pd.DataFrame({
    "ID": test_ids,
    "TargetF1": test_pred.astype(int),
    "TargetRAUC": test_prob
})

submission.head()

,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,0,0.200765
1,ID_TS_NEW_7SPRN3PB,0,0.443070
2,ID_TS_NEW_LZOWPHLC,0,0.143472
3,ID_TS_NEW_DN6TUF64,0,0.309093
4,ID_TS_NEW_95N82M49,0,0.089994


#### Save Submission

In [14]:
submission.to_csv(
    "../submissions/catboost_baseline_submission.csv",
    index=False
)

print("Submission saved successfully!")

Submission saved successfully!
